### NB3 — QSAR Modeling, Internal Benchmark, Cross-Validation, and External Validation
### Author: Hamid Bouseber
### Project: COX-2 QSAR-XAI workflow
### Input: descriptor matrices generated in NB2
### Outputs: benchmark tables, CV results, external validation, final predictions

In [1]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_regression

In [2]:
# ----------------------------
# Configuration
# ----------------------------
MATRICES_DIR = "./matrices"
RESULTS_DIR = "./results"
os.makedirs(RESULTS_DIR, exist_ok=True)

TEST_SIZE = 0.20
RANDOM_STATE = 42
CV_SPLITS = 5

N_ESTIMATORS = 300

DESCRIPTORS = ["pubchem", "morgan", "rdkit"]

# Final model retained for the manuscript
FINAL_DESCRIPTOR = "pubchem"
FINAL_MODEL_NAME = "RandomForest"

In [3]:
def metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return {
        "R2": r2_score(y_true, y_pred),
        "RMSE": np.sqrt(mse),
        "MAE": mean_absolute_error(y_true, y_pred)
    }


def get_standard_models():
    return {
        "DecisionTree": DecisionTreeRegressor(
            random_state=RANDOM_STATE
        ),
        "RandomForest": RandomForestRegressor(
            n_estimators=N_ESTIMATORS,
            random_state=RANDOM_STATE,
            n_jobs=-1
        ),
        "GradientBoosting": GradientBoostingRegressor(
            n_estimators=N_ESTIMATORS,
            random_state=RANDOM_STATE
        )
    }


def get_external_model(descriptor, model_name):
    """
    Models used for final external validation.
    RDKit uses the optimized pipeline identified during testing.
    """
    if descriptor == "rdkit" and model_name == "GB_scaled_SelectK100":
        return Pipeline([
            ("scaler", StandardScaler()),
            ("select", SelectKBest(score_func=f_regression, k=100)),
            ("model", GradientBoostingRegressor(
                n_estimators=N_ESTIMATORS,
                random_state=RANDOM_STATE
            ))
        ])

    if model_name == "DecisionTree":
        return DecisionTreeRegressor(random_state=RANDOM_STATE)

    if model_name == "RandomForest":
        return RandomForestRegressor(
            n_estimators=N_ESTIMATORS,
            random_state=RANDOM_STATE,
            n_jobs=-1
        )

    if model_name == "GradientBoosting":
        return GradientBoostingRegressor(
            n_estimators=N_ESTIMATORS,
            random_state=RANDOM_STATE
        )

    raise ValueError(f"Unknown model: {model_name}")

In [4]:
summary_rows = []

for desc in DESCRIPTORS:
    X_train_test = pd.read_csv(f"{MATRICES_DIR}/X_{desc}_train_test.csv")
    X_external = pd.read_csv(f"{MATRICES_DIR}/X_{desc}_external.csv")

    summary_rows.append({
        "Descriptor": desc,
        "Train/Test samples": X_train_test.shape[0],
        "External samples": X_external.shape[0],
        "Features": X_train_test.shape[1]
    })

df_dataset_summary = pd.DataFrame(summary_rows)

y_train_test = pd.read_csv(f"{MATRICES_DIR}/y_train_test.csv").iloc[:, 0].astype(float)
y_external = pd.read_csv(f"{MATRICES_DIR}/y_external.csv").iloc[:, 0].astype(float)

df_target_summary = pd.DataFrame({
    "Dataset": ["Train/Test", "External"],
    "n": [len(y_train_test), len(y_external)],
    "Mean": [y_train_test.mean(), y_external.mean()],
    "Std": [y_train_test.std(), y_external.std()],
    "Min": [y_train_test.min(), y_external.min()],
    "Median": [y_train_test.median(), y_external.median()],
    "Max": [y_train_test.max(), y_external.max()]
}).round(3)

print("Dataset summary")
display(df_dataset_summary)

print("Target pIC50 summary")
display(df_target_summary)

Dataset summary


,Descriptor,Train/Test samples,External samples,Features
0,pubchem,2297,58,481
1,morgan,2297,58,1020
2,rdkit,2297,58,195


Target pIC50 summary


,Dataset,n,Mean,Std,Min,Median,Max
0,Train/Test,2297,6.322,1.259,1.222,6.456,11.222
1,External,58,6.168,0.600,5.067,6.284,7.328


In [5]:
benchmark_rows = []

for desc in DESCRIPTORS:
    X = pd.read_csv(f"{MATRICES_DIR}/X_{desc}_train_test.csv")
    y = pd.read_csv(f"{MATRICES_DIR}/y_train_test.csv").iloc[:, 0].astype(float)

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        shuffle=True
    )

    for model_name, model in get_standard_models().items():
        model.fit(X_train, y_train)

        pred_train = model.predict(X_train)
        pred_test = model.predict(X_test)

        m_train = metrics(y_train, pred_train)
        m_test = metrics(y_test, pred_test)

        benchmark_rows.append({
            "Descriptor": desc,
            "Model": model_name,
            "Train_R2": m_train["R2"],
            "Train_RMSE": m_train["RMSE"],
            "Train_MAE": m_train["MAE"],
            "Test_R2": m_test["R2"],
            "Test_RMSE": m_test["RMSE"],
            "Test_MAE": m_test["MAE"],
            "Overfit_gap": m_train["R2"] - m_test["R2"]
        })

df_table1 = pd.DataFrame(benchmark_rows)

df_table1_display = (
    df_table1
    .sort_values(by=["Test_R2", "Test_RMSE"], ascending=[False, True])
    .reset_index(drop=True)
    .round(3)
)

df_table1.to_csv(f"{RESULTS_DIR}/Table1_internal_benchmark.csv", index=False)

df_table1_display

,Descriptor,Model,Train_R2,Train_RMSE,Train_MAE,Test_R2,Test_RMSE,Test_MAE,Overfit_gap
0,pubchem,RandomForest,0.928,0.336,0.225,0.653,0.753,0.527,0.275
1,rdkit,GradientBoosting,0.874,0.445,0.340,0.638,0.769,0.568,0.236
2,morgan,RandomForest,0.938,0.312,0.210,0.636,0.771,0.530,0.302
3,rdkit,RandomForest,0.940,0.306,0.214,0.621,0.787,0.556,0.319
4,pubchem,GradientBoosting,0.759,0.615,0.459,0.620,0.787,0.589,0.139
5,morgan,GradientBoosting,0.799,0.562,0.436,0.612,0.796,0.579,0.187
6,pubchem,DecisionTree,0.967,0.229,0.068,0.379,1.007,0.649,0.588
7,morgan,DecisionTree,0.984,0.159,0.028,0.335,1.042,0.717,0.649
8,rdkit,DecisionTree,0.988,0.138,0.017,0.335,1.042,0.695,0.653


In [6]:
X_cv = pd.read_csv(f"{MATRICES_DIR}/X_{FINAL_DESCRIPTOR}_train_test.csv")
y_cv = pd.read_csv(f"{MATRICES_DIR}/y_train_test.csv").iloc[:, 0].astype(float)

kf = KFold(
    n_splits=CV_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

cv_rows = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X_cv), start=1):
    X_train_fold = X_cv.iloc[train_idx]
    X_val_fold = X_cv.iloc[val_idx]
    y_train_fold = y_cv.iloc[train_idx]
    y_val_fold = y_cv.iloc[val_idx]

    model = RandomForestRegressor(
        n_estimators=N_ESTIMATORS,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    model.fit(X_train_fold, y_train_fold)
    pred_val = model.predict(X_val_fold)

    m = metrics(y_val_fold, pred_val)

    cv_rows.append({
        "Fold": fold,
        "R2": m["R2"],
        "RMSE": m["RMSE"],
        "MAE": m["MAE"]
    })

df_cv_folds = pd.DataFrame(cv_rows)

mean_row = {
    "Fold": "Mean",
    "R2": df_cv_folds["R2"].mean(),
    "RMSE": df_cv_folds["RMSE"].mean(),
    "MAE": df_cv_folds["MAE"].mean()
}

std_row = {
    "Fold": "SD",
    "R2": df_cv_folds["R2"].std(),
    "RMSE": df_cv_folds["RMSE"].std(),
    "MAE": df_cv_folds["MAE"].std()
}

df_table2 = pd.concat(
    [df_cv_folds, pd.DataFrame([mean_row, std_row])],
    ignore_index=True
)

df_cv_folds.to_csv(f"{RESULTS_DIR}/Table2_CV_folds_final_model.csv", index=False)
df_table2.to_csv(f"{RESULTS_DIR}/Table2_CV_summary_final_model.csv", index=False)

df_table2.round(3)

,Fold,R2,RMSE,MAE
0,1,0.649,0.756,0.532
1,2,0.736,0.685,0.491
2,3,0.590,0.771,0.547
3,4,0.566,0.767,0.530
4,5,0.655,0.764,0.552
5,Mean,0.639,0.749,0.530
6,SD,0.066,0.036,0.024


In [7]:
# Best models retained for external validation
# PubChem and Morgan use the best standard model.
# RDKit uses the improved scaled + feature-selected pipeline.

BEST_MODELS_EXTERNAL = {
    "pubchem": "RandomForest",
    "morgan": "RandomForest",
    "rdkit": "GB_scaled_SelectK100"
}

In [8]:
external_rows = []

y_full = pd.read_csv(f"{MATRICES_DIR}/y_train_test.csv").iloc[:, 0].astype(float)
y_ext = pd.read_csv(f"{MATRICES_DIR}/y_external.csv").iloc[:, 0].astype(float)

for desc, model_name in BEST_MODELS_EXTERNAL.items():
    X_full = pd.read_csv(f"{MATRICES_DIR}/X_{desc}_train_test.csv")
    X_ext = pd.read_csv(f"{MATRICES_DIR}/X_{desc}_external.csv")

    # Internal test R2 from Table 1 when available
    if model_name in df_table1[df_table1["Descriptor"] == desc]["Model"].values:
        test_r2 = df_table1[
            (df_table1["Descriptor"] == desc) &
            (df_table1["Model"] == model_name)
        ]["Test_R2"].iloc[0]
    else:
        # Compute internal test R2 for the optimized RDKit model
        X_train, X_test, y_train, y_test = train_test_split(
            X_full,
            y_full,
            test_size=TEST_SIZE,
            random_state=RANDOM_STATE,
            shuffle=True
        )

        tmp_model = get_external_model(desc, model_name)
        tmp_model.fit(X_train, y_train)
        test_r2 = metrics(y_test, tmp_model.predict(X_test))["R2"]

    # Final external validation: refit on 100% train/test
    final_model = get_external_model(desc, model_name)
    final_model.fit(X_full, y_full)

    pred_ext = final_model.predict(X_ext)
    m_ext = metrics(y_ext, pred_ext)

    external_rows.append({
        "Descriptor": desc,
        "Best_Model": model_name,
        "Test_R2": test_r2,
        "External_n": len(y_ext),
        "External_R2": m_ext["R2"],
        "External_RMSE": m_ext["RMSE"],
        "External_MAE": m_ext["MAE"]
    })

df_table3 = pd.DataFrame(external_rows)

df_table3_display = (
    df_table3
    .sort_values(by="External_R2", ascending=False)
    .reset_index(drop=True)
    .round(3)
)

df_table3.to_csv(f"{RESULTS_DIR}/Table3_external_validation.csv", index=False)

df_table3_display

,Descriptor,Best_Model,Test_R2,External_n,External_R2,External_RMSE,External_MAE
0,pubchem,RandomForest,0.653,58,0.509,0.416,0.350
1,morgan,RandomForest,0.636,58,0.468,0.433,0.351
2,rdkit,GB_scaled_SelectK100,0.613,58,0.357,0.477,0.413


In [10]:
X_full = pd.read_csv(f"{MATRICES_DIR}/X_pubchem_train_test.csv")
X_ext = pd.read_csv(f"{MATRICES_DIR}/X_pubchem_external.csv")

y_full = pd.read_csv(f"{MATRICES_DIR}/y_train_test.csv").iloc[:, 0].astype(float)
y_ext = pd.read_csv(f"{MATRICES_DIR}/y_external.csv").iloc[:, 0].astype(float)

final_model = RandomForestRegressor(
    n_estimators=N_ESTIMATORS,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

final_model.fit(X_full, y_full)

pred_full = final_model.predict(X_full)
pred_ext = final_model.predict(X_ext)

internal_predictions = pd.DataFrame({
    "dataset": "train_test_full",
    "pIC50_exp": y_full.values,
    "pIC50_pred": pred_full
})

external_predictions = pd.DataFrame({
    "dataset": "external",
    "pIC50_exp": y_ext.values,
    "pIC50_pred": pred_ext
})

final_predictions = pd.concat(
    [internal_predictions, external_predictions],
    ignore_index=True
)

internal_predictions.to_csv(
    f"{RESULTS_DIR}/internal_predictions_final_model.csv",
    index=False
)

external_predictions.to_csv(
    f"{RESULTS_DIR}/external_predictions_final_model.csv",
    index=False
)

final_predictions.to_csv(
    f"{RESULTS_DIR}/final_model_predictions_all.csv",
    index=False
)

final_model_summary = pd.DataFrame([{
    "Descriptor": "pubchem",
    "Model": "RandomForest",
    "Full_train_R2": metrics(y_full, pred_full)["R2"],
    "Full_train_RMSE": metrics(y_full, pred_full)["RMSE"],
    "Full_train_MAE": metrics(y_full, pred_full)["MAE"],
    "External_R2": metrics(y_ext, pred_ext)["R2"],
    "External_RMSE": metrics(y_ext, pred_ext)["RMSE"],
    "External_MAE": metrics(y_ext, pred_ext)["MAE"],
    "External_n": len(y_ext)
}])

final_model_summary.to_csv(
    f"{RESULTS_DIR}/selected_model_summary.csv",
    index=False
)

final_model_summary.round(3)

,Descriptor,Model,Full_train_R2,Full_train_RMSE,Full_train_MAE,External_R2,External_RMSE,External_MAE,External_n
0,pubchem,RandomForest,0.926,0.342,0.225,0.509,0.416,0.35,58


In [11]:
required_outputs = [
    "Table1_internal_benchmark.csv",
    "Table2_CV_folds_final_model.csv",
    "Table2_CV_summary_final_model.csv",
    "Table3_external_validation.csv",
    "internal_predictions_final_model.csv",
    "external_predictions_final_model.csv",
    "final_model_predictions_all.csv",
    "selected_model_summary.csv",
]

for file in required_outputs:
    path = f"{RESULTS_DIR}/{file}"
    assert os.path.exists(path), f"Missing output: {path}"

print("NB3 completed successfully.")
print("Tables and prediction files saved in:", RESULTS_DIR)

NB3 completed successfully.
Tables and prediction files saved in: ./results
